# Solar PPA microgrid

A solar-plus-storage microgrid with a PPA revenue contract, degradation, O&M escalation, ITC/PTC tax attributes and MACRS depreciation, financed with sculpted debt.

This notebook uses the benchmark model that CFDL validates against an independent reference to the penny (see `benchmarks/`).

In [ ]:
from pathlib import Path
import cfdl_sdk

# Resolve the repo root so the notebook runs from anywhere in a checkout.
ROOT = Path.cwd()
while not (ROOT / "Cargo.toml").exists():
    ROOT = ROOT.parent
PACKS = ROOT / "packs"

## Compile

Compile the model directory to IR.

In [ ]:
model_dir = ROOT / "benchmarks/energy/solar_ppa_microgrid"
model = cfdl_sdk.compile(model_dir, packs_dir=PACKS)
print("streams:", len(model.ir["streams"]))

## Run

Run with the benchmark's configuration and apply the `energy` pack's domain metrics.

In [ ]:
results = model.run(
    config=str(model_dir / "run.json"),
    pack="energy",
)
print("status:", results.status, "| warnings:", len(results.warnings))

## Cash flows

The engine returns per-period signed cash flows; `cashflows()` gives a wide DataFrame indexed by period.

In [ ]:
cf = results.cashflows()
print('shape:', cf.shape)
cf.head()

In [ ]:
# Requires the [viz] extra (pip install cfdl-sdk[viz]).
results.plot.cumulative()

## Metrics

Core metrics (NPV/IRR/MOIC/...) plus the pack's domain metrics, with their source labelled.

In [ ]:
results.metrics_frame()

## What-if

Re-run at a higher discount rate and compare NPV.

In [ ]:
base = results.metrics()["model.npv"]
stressed = model.run(config={"deterministic": {"annual_discount_rate": 0.10}}, pack="energy")
print(f"NPV @ base: {base:,.0f}")
print(f"NPV @ 10%: {stressed.metrics()['model.npv']:,.0f}")